# Proyecto #2 - Data Mining (PBL)
## Semana 3: Validacion, Optimizacion y Comparacion Final

---

## 1. Importacion de librerias

In [16]:
%pip install matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_JOBS = 1
LABELS_CLASE = [2, 4]

print('Librerias cargadas correctamente.')

Librerias cargadas correctamente.


## 2. Confirmacion del dataset final y split heredado

In [18]:
candidatos = [
    Path('data/Datos_modelado.csv'),
    Path('../data/Datos_modelado.csv'),
    Path('Datos_modelado.csv')
]
ruta_datos = next((p for p in candidatos if p.exists()), None)
if ruta_datos is None:
    raise FileNotFoundError('No se encontro Datos_modelado.csv en rutas esperadas.')

df = pd.read_csv(ruta_datos)
df.columns = df.columns.str.strip()

expected_rows = 675
expected_cols = 10

print('=' * 90)
print('VALIDACION DE INSUMOS DE SEMANA 2')
print('=' * 90)
print(f'Archivo detectado: {ruta_datos}')
print(f'Dimension actual : {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Esperado S2      : {expected_rows} filas x {expected_cols} columnas')
print(f'Coincide?        : {df.shape == (expected_rows, expected_cols)}')
print('\nDistribucion de clases:')
print(df['Class'].value_counts().sort_index())
print('=' * 90)

VALIDACION DE INSUMOS DE SEMANA 2
Archivo detectado: ..\data\Datos_modelado.csv
Dimension actual : 675 filas x 10 columnas
Esperado S2      : 675 filas x 10 columnas
Coincide?        : True

Distribucion de clases:
Class
2    439
4    236
Name: count, dtype: int64


In [19]:
target_col = 'Class'
id_col = 'Sample code number'

if id_col in df.columns:
    feature_cols = [c for c in df.columns if c not in [id_col, target_col]]
else:
    feature_cols = [c for c in df.columns if c != target_col]

X_all = df[feature_cols].values
y = df[target_col].values

top3_features = ['Bare Nuclei', 'Uniformity of Cell Size', 'Uniformity of Cell Shape']
if not set(top3_features).issubset(set(feature_cols)):
    raise ValueError('No se encontraron todas las variables del escenario Top-3.')

X_top = df[top3_features].values

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

X_train_top, X_test_top, _, _ = train_test_split(
    X_top, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

escenarios = {
    'A - Todas las variables': (X_train_all, X_test_all),
    'B - Top 3 variables': (X_train_top, X_test_top),
}

def ratio_clases(arr):
    s = pd.Series(arr).value_counts(normalize=True).sort_index()
    return s

print('=' * 90)
print('CONFIRMACION DEL SPLIT (S2 -> S3)')
print('=' * 90)
print(f'X_train_all: {X_train_all.shape} | X_test_all: {X_test_all.shape}')
print(f'X_train_top: {X_train_top.shape} | X_test_top: {X_test_top.shape}')
print(f'y_train    : {y_train.shape}      | y_test    : {y_test.shape}')

dist_total = ratio_clases(y)
dist_train = ratio_clases(y_train)
dist_test = ratio_clases(y_test)

tabla_dist = pd.DataFrame({
    'Total': dist_total,
    'Train': dist_train,
    'Test': dist_test
}).fillna(0).round(4)

print('\nProporcion de clases (2=Benigno, 4=Maligno):')
print(tabla_dist)
print('=' * 90)

CONFIRMACION DEL SPLIT (S2 -> S3)
X_train_all: (540, 9) | X_test_all: (135, 9)
X_train_top: (540, 3) | X_test_top: (135, 3)
y_train    : (540,)      | y_test    : (135,)

Proporcion de clases (2=Benigno, 4=Maligno):
    Total  Train    Test
2  0.6504   0.65  0.6519
4  0.3496   0.35  0.3481


## 3. Metodologia de validacion

**Esquema adoptado: 5-Fold Stratified Cross-Validation**

Criterios del diseno experimental:
1. Mantener proporcion de clases en cada fold.
2. Evaluar todos los modelos bajo condiciones identicas.
3. Reportar media y desviacion estandar de metricas por modelo y escenario.
4. Mantener el test set virgen para la verificacion final.

In [20]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision_weighted',
    'recall_macro': 'recall_macro',
    'f1': 'f1_weighted',
    'roc_auc': 'roc_auc_ovr_weighted'
}

modelos_base = {
    'Regresion Logistica': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'KNN (k=5)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE, probability=True))
    ])
}

print('=' * 90)
print('CONFIGURACION DE VALIDACION')
print('=' * 90)
print('Estrategia: 5-Fold Stratified CV')
print(f'Folds: {cv_strategy.get_n_splits()}')
print(f'Metricas: {list(scoring_metrics.keys())}')
print(f'Modelos: {list(modelos_base.keys())}')
print('=' * 90)

CONFIGURACION DE VALIDACION
Estrategia: 5-Fold Stratified CV
Folds: 5
Metricas: ['accuracy', 'precision', 'recall_macro', 'f1', 'roc_auc']
Modelos: ['Regresion Logistica', 'KNN (k=5)', 'SVM (RBF)']


## 4. Ejecucion de validacion base (pre-tuning)

In [21]:
cv_results_store = {}

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    print('=' * 90)
    print(f'VALIDACION CRUZADA | ESCENARIO: {nombre_escenario}')
    print('=' * 90)

    for modelo_nombre, modelo_pipeline in modelos_base.items():
        print(f'\n>>> {modelo_nombre} | {nombre_escenario}')

        cv_results = cross_validate(
            modelo_pipeline,
            X_tr_esc,
            y_train,
            cv=cv_strategy,
            scoring=scoring_metrics,
            n_jobs=N_JOBS,
            return_train_score=False
        )

        key = f'{modelo_nombre} | {nombre_escenario}'
        cv_results_store[key] = cv_results

        for metrica in ['accuracy', 'precision', 'recall_macro', 'f1', 'roc_auc']:
            vals = cv_results[f'test_{metrica}']
            print(f'   {metrica.upper():15}: {vals.mean():.4f} +/- {vals.std():.4f}')

print('\n' + '=' * 90)
print('VALIDACION BASE COMPLETADA')
print('=' * 90)

VALIDACION CRUZADA | ESCENARIO: A - Todas las variables

>>> Regresion Logistica | A - Todas las variables
   ACCURACY       : 0.9704 +/- 0.0283
   PRECISION      : 0.9705 +/- 0.0284
   RECALL_MACRO   : 0.9664 +/- 0.0351
   F1             : 0.9702 +/- 0.0287
   ROC_AUC        : 0.9941 +/- 0.0074

>>> KNN (k=5) | A - Todas las variables
   ACCURACY       : 0.9685 +/- 0.0224
   PRECISION      : 0.9685 +/- 0.0225
   RECALL_MACRO   : 0.9636 +/- 0.0272
   F1             : 0.9684 +/- 0.0226
   ROC_AUC        : 0.9843 +/- 0.0262

>>> SVM (RBF) | A - Todas las variables
   ACCURACY       : 0.9611 +/- 0.0312
   PRECISION      : 0.9624 +/- 0.0310
   RECALL_MACRO   : 0.9617 +/- 0.0341
   F1             : 0.9613 +/- 0.0312
   ROC_AUC        : 0.9898 +/- 0.0109
VALIDACION CRUZADA | ESCENARIO: B - Top 3 variables

>>> Regresion Logistica | B - Top 3 variables
   ACCURACY       : 0.9648 +/- 0.0271
   PRECISION      : 0.9648 +/- 0.0272
   RECALL_MACRO   : 0.9583 +/- 0.0305
   F1             : 0.9647 +

## 5. Tablas de resultados de cross-validation

In [22]:
rows_cv = []

for key, results in cv_results_store.items():
    modelo_nombre, escenario = key.split(' | ')
    rows_cv.append({
        'Modelo': modelo_nombre,
        'Escenario': escenario,
        'Accuracy (mean+/-std)': f"{results['test_accuracy'].mean():.4f} +/- {results['test_accuracy'].std():.4f}",
        'Precision (mean+/-std)': f"{results['test_precision'].mean():.4f} +/- {results['test_precision'].std():.4f}",
        'Recall (mean+/-std)': f"{results['test_recall_macro'].mean():.4f} +/- {results['test_recall_macro'].std():.4f}",
        'F1 (mean+/-std)': f"{results['test_f1'].mean():.4f} +/- {results['test_f1'].std():.4f}",
        'AUC (mean+/-std)': f"{results['test_roc_auc'].mean():.4f} +/- {results['test_roc_auc'].std():.4f}"
    })

df_cv_results = pd.DataFrame(rows_cv)

print('=' * 140)
print('TABLA GLOBAL: RESULTADOS 5-FOLD STRATIFIED CV')
print('=' * 140)
print(df_cv_results.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA POR ESCENARIO A - TODAS LAS VARIABLES')
print('=' * 140)
print(df_cv_results[df_cv_results['Escenario'] == 'A - Todas las variables'].drop(columns=['Escenario']).to_string(index=False))

print('\n' + '=' * 140)
print('TABLA POR ESCENARIO B - TOP 3 VARIABLES')
print('=' * 140)
print(df_cv_results[df_cv_results['Escenario'] == 'B - Top 3 variables'].drop(columns=['Escenario']).to_string(index=False))

TABLA GLOBAL: RESULTADOS 5-FOLD STRATIFIED CV
             Modelo               Escenario Accuracy (mean+/-std) Precision (mean+/-std) Recall (mean+/-std)   F1 (mean+/-std)  AUC (mean+/-std)
Regresion Logistica A - Todas las variables     0.9704 +/- 0.0283      0.9705 +/- 0.0284   0.9664 +/- 0.0351 0.9702 +/- 0.0287 0.9941 +/- 0.0074
          KNN (k=5) A - Todas las variables     0.9685 +/- 0.0224      0.9685 +/- 0.0225   0.9636 +/- 0.0272 0.9684 +/- 0.0226 0.9843 +/- 0.0262
          SVM (RBF) A - Todas las variables     0.9611 +/- 0.0312      0.9624 +/- 0.0310   0.9617 +/- 0.0341 0.9613 +/- 0.0312 0.9898 +/- 0.0109
Regresion Logistica     B - Top 3 variables     0.9648 +/- 0.0271      0.9648 +/- 0.0272   0.9583 +/- 0.0305 0.9647 +/- 0.0272 0.9912 +/- 0.0112
          KNN (k=5)     B - Top 3 variables     0.9667 +/- 0.0239      0.9671 +/- 0.0234   0.9634 +/- 0.0246 0.9667 +/- 0.0237 0.9823 +/- 0.0167
          SVM (RBF)     B - Top 3 variables     0.9574 +/- 0.0319      0.9577 +/- 0.

In [23]:
# Comparacion CV vs Test para confirmar consistencia de generalizacion
test_rows = []

for nombre_esc, (X_tr_esc, X_te_esc) in escenarios.items():
    for modelo_nombre, modelo_pipeline in modelos_base.items():
        modelo_pipeline.fit(X_tr_esc, y_train)
        y_pred = modelo_pipeline.predict(X_te_esc)
        y_proba = modelo_pipeline.predict_proba(X_te_esc)[:, 1]

        acc_test = accuracy_score(y_test, y_pred)
        rec_test = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1_test = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        auc_test = roc_auc_score(y_test, y_proba, labels=LABELS_CLASE)

        key = f'{modelo_nombre} | {nombre_esc}'
        acc_cv = cv_results_store[key]['test_accuracy'].mean()
        rec_cv = cv_results_store[key]['test_recall_macro'].mean()
        f1_cv = cv_results_store[key]['test_f1'].mean()
        auc_cv = cv_results_store[key]['test_roc_auc'].mean()

        test_rows.append({
            'Modelo': modelo_nombre,
            'Escenario': nombre_esc,
            'Acc_CV': round(acc_cv, 4),
            'Acc_Test': round(acc_test, 4),
            'Gap_Acc': round(acc_test - acc_cv, 4),
            'Recall_CV': round(rec_cv, 4),
            'Recall_Test': round(rec_test, 4),
            'Gap_Recall': round(rec_test - rec_cv, 4),
            'F1_CV': round(f1_cv, 4),
            'F1_Test': round(f1_test, 4),
            'AUC_CV': round(auc_cv, 4),
            'AUC_Test': round(auc_test, 4)
        })

df_cv_vs_test = pd.DataFrame(test_rows)
print('=' * 140)
print('TABLA DE CONSISTENCIA: CV VS TEST')
print('=' * 140)
print(df_cv_vs_test.to_string(index=False))

TABLA DE CONSISTENCIA: CV VS TEST
             Modelo               Escenario  Acc_CV  Acc_Test  Gap_Acc  Recall_CV  Recall_Test  Gap_Recall  F1_CV  F1_Test  AUC_CV  AUC_Test
Regresion Logistica A - Todas las variables  0.9704    0.9556  -0.0148     0.9664       0.9510     -0.0153 0.9702   0.9556  0.9941    0.9952
          KNN (k=5) A - Todas las variables  0.9685    0.9556  -0.0130     0.9636       0.9510     -0.0126 0.9684   0.9556  0.9843    0.9900
          SVM (RBF) A - Todas las variables  0.9611    0.9630   0.0019     0.9617       0.9617     -0.0000 0.9613   0.9631  0.9898    0.9908
Regresion Logistica     B - Top 3 variables  0.9648    0.9259  -0.0389     0.9583       0.9085     -0.0498 0.9647   0.9251  0.9912    0.9915
          KNN (k=5)     B - Top 3 variables  0.9667    0.9481  -0.0185     0.9634       0.9404     -0.0230 0.9667   0.9480  0.9823    0.9840
          SVM (RBF)     B - Top 3 variables  0.9574    0.9481  -0.0093     0.9538       0.9404     -0.0134 0.9575   0.94

## 6. Control de fuga de informacion

Reglas aplicadas en este flujo:
- El escalado se realiza dentro de Pipeline.
- Cada fold ajusta scaler/modelo solo con datos de entrenamiento del fold.
- El test set no participa en CV ni en decisiones de seleccion.
- La evaluacion en test se realiza al final para confirmar generalizacion.

Checklist:
- [x] Separacion train/test estratificada conservada.
- [x] CV estratificada para mantener proporcion de clases.
- [x] Condiciones identicas para todos los modelos y escenarios.
- [x] Tabla de resultados por modelo y por escenario.
- [x] Documento breve del metodo de validacion.

## 7. Texto breve del metodo de validacion usado

Se utilizo un esquema de 5-Fold Stratified Cross-Validation sobre el conjunto de entrenamiento heredado de Semana 2 (split 80/20 estratificado con random_state=42). La estratificacion aseguro mantener la proporcion de clases en cada fold, reduciendo sesgos por desbalance. Todos los modelos base (Regresion Logistica, KNN y SVM RBF) fueron evaluados bajo las mismas metricas y con el mismo protocolo, para garantizar comparabilidad. El conjunto de prueba se mantuvo intacto hasta el final para verificar consistencia y evitar fuga de informacion.

## 8. Tuning de hiperparametros: Regresion Logistica y KNN

Objetivo de esta seccion:
- Optimizar LR y KNN con el mismo esquema de validacion de la seccion anterior.
- Usar como metrica principal el recall de la clase maligna (Class=4).
- Repetir el tuning en ambos escenarios de variables para comparacion formal.

In [24]:
# metrica principal para seleccion: recall de clase maligna (Class=4)
scorer_recall_maligna = make_scorer(recall_score, pos_label=4, zero_division=0)

# grilla media para regresion logistica (combinaciones validas por solver)
param_grid_lr = [
    {
        'model__solver': ['lbfgs'],
        'model__penalty': ['l2'],
        'model__C': [0.01, 0.1, 1, 3, 10, 30, 100],
        'model__class_weight': [None, 'balanced'],
        'model__max_iter': [2000]
    },
    {
        'model__solver': ['liblinear'],
        'model__penalty': ['l1', 'l2'],
        'model__C': [0.01, 0.1, 1, 3, 10, 30, 100],
        'model__class_weight': [None, 'balanced'],
        'model__max_iter': [2000]
    },
    {
        'model__solver': ['saga'],
        'model__penalty': ['l1', 'l2'],
        'model__C': [0.01, 0.1, 1, 3, 10, 30, 100],
        'model__class_weight': [None, 'balanced'],
        'model__max_iter': [3000]
    }
]

# grilla media para knn
param_grid_knn = [
    {
        'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 25],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['euclidean', 'manhattan']
    },
    {
        'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 25],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['minkowski'],
        'model__p': [1, 2]
    }
]

base_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

base_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

print('=' * 90)
print('CONFIGURACION DE TUNING (LR + KNN)')
print('=' * 90)
print('Metrica principal de seleccion: recall_maligna (pos_label=4)')
print(f'CV: StratifiedKFold con {cv_strategy.get_n_splits()} folds')
print(f'Combinaciones LR aproximadas: {sum(len(d["model__C"]) * len(d["model__class_weight"]) * len(d["model__penalty"]) * len(d["model__solver"]) for d in param_grid_lr)}')
print(f'Combinaciones KNN aproximadas: {8*2*2 + 8*2*1*2}')
print('=' * 90)

CONFIGURACION DE TUNING (LR + KNN)
Metrica principal de seleccion: recall_maligna (pos_label=4)
CV: StratifiedKFold con 5 folds
Combinaciones LR aproximadas: 70
Combinaciones KNN aproximadas: 64


In [25]:
tuning_store = {}
rows_best = []

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    print('\n' + '=' * 90)
    print(f'TUNING EN ESCENARIO: {nombre_escenario}')
    print('=' * 90)

    # LR
    grid_lr = GridSearchCV(
        estimator=base_lr,
        param_grid=param_grid_lr,
        scoring=scorer_recall_maligna,
        cv=cv_strategy,
        n_jobs=N_JOBS,
        refit=True,
        return_train_score=True
    )
    grid_lr.fit(X_tr_esc, y_train)

    # KNN
    grid_knn = GridSearchCV(
        estimator=base_knn,
        param_grid=param_grid_knn,
        scoring=scorer_recall_maligna,
        cv=cv_strategy,
        n_jobs=N_JOBS,
        refit=True,
        return_train_score=True
    )
    grid_knn.fit(X_tr_esc, y_train)

    tuning_store[(nombre_escenario, 'Regresion Logistica')] = grid_lr
    tuning_store[(nombre_escenario, 'KNN')] = grid_knn

    for modelo_nombre, grid_obj in [('Regresion Logistica', grid_lr), ('KNN', grid_knn)]:
        best_idx = grid_obj.best_index_
        best_mean = grid_obj.cv_results_['mean_test_score'][best_idx]
        best_std = grid_obj.cv_results_['std_test_score'][best_idx]

        rows_best.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Best Recall CV (maligna)': round(best_mean, 4),
            'Std Recall CV': round(best_std, 4),
            'Best Params': str(grid_obj.best_params_)
        })

        print(f"{modelo_nombre:22s} | Best Recall CV={best_mean:.4f} +/- {best_std:.4f}")

best_params_df = pd.DataFrame(rows_best)

print('\n' + '=' * 120)
print('TABLA: MEJORES HIPERPARAMETROS (LR + KNN)')
print('=' * 120)
print(best_params_df.to_string(index=False))


TUNING EN ESCENARIO: A - Todas las variables
Regresion Logistica    | Best Recall CV=0.9789 +/- 0.0421
KNN                    | Best Recall CV=0.9632 +/- 0.0488

TUNING EN ESCENARIO: B - Top 3 variables
Regresion Logistica    | Best Recall CV=0.9471 +/- 0.0333
KNN                    | Best Recall CV=0.9578 +/- 0.0268

TABLA: MEJORES HIPERPARAMETROS (LR + KNN)
              Escenario              Modelo  Best Recall CV (maligna)  Std Recall CV                                                                                                                         Best Params
A - Todas las variables Regresion Logistica                    0.9789         0.0421 {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 2000, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
A - Todas las variables                 KNN                    0.9632         0.0488                                                {'model__metric': 'euclidean', 'model__n_neighbors': 9, 'model__weights

In [26]:
def evaluar_en_test(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    if hasattr(modelo, 'predict_proba'):
        y_score = modelo.predict_proba(X_test)[:, 1]
    else:
        y_score = None

    out = {
        'accuracy_test': accuracy_score(y_test, y_pred),
        'precision_maligna_test': precision_score(y_test, y_pred, pos_label=4, zero_division=0),
        'recall_maligna_test': recall_score(y_test, y_pred, pos_label=4, zero_division=0),
        'f1_maligna_test': f1_score(y_test, y_pred, pos_label=4, zero_division=0),
        'fn_test': int(((y_test == 4) & (y_pred != 4)).sum())
    }

    if y_score is not None:
        out['auc_test'] = roc_auc_score(y_test, y_score, labels=LABELS_CLASE)
    else:
        out['auc_test'] = np.nan

    return out

rows_compare = []
rows_fit_diag = []

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    for modelo_nombre in ['Regresion Logistica', 'KNN']:
        base_model = modelos_base['Regresion Logistica'] if modelo_nombre == 'Regresion Logistica' else modelos_base['KNN (k=5)']
        tuned_grid = tuning_store[(nombre_escenario, modelo_nombre)]
        tuned_model = tuned_grid.best_estimator_

        # base metrics bajo la misma metrica objetivo del tuning
        base_cv_obj = cross_validate(
            base_model,
            X_tr_esc,
            y_train,
            cv=cv_strategy,
            scoring=scorer_recall_maligna,
            n_jobs=N_JOBS,
            return_train_score=True
        )
        base_cv_recall = base_cv_obj['test_score'].mean()

        base_test = evaluar_en_test(base_model.fit(X_tr_esc, y_train), X_te_esc, y_test)

        # tuned metrics (CV principal + test)
        tuned_best_idx = tuned_grid.best_index_
        tuned_cv_recall = tuned_grid.cv_results_['mean_test_score'][tuned_best_idx]
        tuned_cv_train_recall = tuned_grid.cv_results_['mean_train_score'][tuned_best_idx]
        tuned_gap = tuned_cv_train_recall - tuned_cv_recall
        tuned_test = evaluar_en_test(tuned_model, X_te_esc, y_test)

        rows_compare.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Tipo': 'Base',
            'Recall_CV_objetivo': round(base_cv_recall, 4),
            'Recall_Test_maligna': round(base_test['recall_maligna_test'], 4),
            'Precision_Test_maligna': round(base_test['precision_maligna_test'], 4),
            'F1_Test_maligna': round(base_test['f1_maligna_test'], 4),
            'Accuracy_Test': round(base_test['accuracy_test'], 4),
            'AUC_Test': round(base_test['auc_test'], 4),
            'FN_Test': base_test['fn_test']
        })

        rows_compare.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Tipo': 'Optimizado',
            'Recall_CV_objetivo': round(tuned_cv_recall, 4),
            'Recall_Test_maligna': round(tuned_test['recall_maligna_test'], 4),
            'Precision_Test_maligna': round(tuned_test['precision_maligna_test'], 4),
            'F1_Test_maligna': round(tuned_test['f1_maligna_test'], 4),
            'Accuracy_Test': round(tuned_test['accuracy_test'], 4),
            'AUC_Test': round(tuned_test['auc_test'], 4),
            'FN_Test': tuned_test['fn_test']
        })

        rows_fit_diag.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Recall_train_CV_best': round(tuned_cv_train_recall, 4),
            'Recall_valid_CV_best': round(tuned_cv_recall, 4),
            'Gap_train_valid': round(tuned_gap, 4),
            'Diagnostico_ajuste': 'posible sobreajuste' if tuned_gap > 0.05 else ('posible subajuste' if tuned_cv_recall < 0.85 else 'estable')
        })

df_compare = pd.DataFrame(rows_compare)
df_fit_diag = pd.DataFrame(rows_fit_diag)

# deltas base vs optimizado
rows_delta = []
for (esc, mod), grp in df_compare.groupby(['Escenario', 'Modelo']):
    base_row = grp[grp['Tipo'] == 'Base'].iloc[0]
    opt_row = grp[grp['Tipo'] == 'Optimizado'].iloc[0]
    rows_delta.append({
        'Escenario': esc,
        'Modelo': mod,
        'Delta_Recall_CV_objetivo': round(opt_row['Recall_CV_objetivo'] - base_row['Recall_CV_objetivo'], 4),
        'Delta_Recall_Test_maligna': round(opt_row['Recall_Test_maligna'] - base_row['Recall_Test_maligna'], 4),
        'Delta_Precision_Test_maligna': round(opt_row['Precision_Test_maligna'] - base_row['Precision_Test_maligna'], 4),
        'Delta_F1_Test_maligna': round(opt_row['F1_Test_maligna'] - base_row['F1_Test_maligna'], 4),
        'Delta_Accuracy_Test': round(opt_row['Accuracy_Test'] - base_row['Accuracy_Test'], 4),
        'Delta_AUC_Test': round(opt_row['AUC_Test'] - base_row['AUC_Test'], 4),
        'Delta_FN_Test': int(opt_row['FN_Test'] - base_row['FN_Test'])
    })

df_delta = pd.DataFrame(rows_delta)

print('=' * 140)
print('TABLA: COMPARACION BASE VS OPTIMIZADO (LR + KNN)')
print('=' * 140)
print(df_compare.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA: DELTAS DE MEJORA (OPTIMIZADO - BASE)')
print('=' * 140)
print(df_delta.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA: ANALISIS DE AJUSTE (SENALES DE SOBREAJUSTE/SUBAJUSTE)')
print('=' * 140)
print(df_fit_diag.to_string(index=False))

TABLA: COMPARACION BASE VS OPTIMIZADO (LR + KNN)
              Escenario              Modelo       Tipo  Recall_CV_objetivo  Recall_Test_maligna  Precision_Test_maligna  F1_Test_maligna  Accuracy_Test  AUC_Test  FN_Test
A - Todas las variables Regresion Logistica       Base              0.9526               0.9362                  0.9362           0.9362         0.9556    0.9952        3
A - Todas las variables Regresion Logistica Optimizado              0.9789               0.9574                  0.9375           0.9474         0.9630    0.9944        2
A - Todas las variables                 KNN       Base              0.9472               0.9362                  0.9362           0.9362         0.9556    0.9900        3
A - Todas las variables                 KNN Optimizado              0.9632               0.9149                  0.9348           0.9247         0.9481    0.9889        4
    B - Top 3 variables Regresion Logistica       Base              0.9366               0.8511 

## 9. Resumen tecnico de tuning

- Se optimizaron Regresion Logistica y KNN en ambos escenarios de variables usando GridSearchCV.
- La metrica principal para seleccion fue el recall de la clase maligna (Class=4), priorizando reduccion de falsos negativos.
- La comparacion base vs optimizado se reporta en CV y en test para confirmar si la mejora se sostiene fuera de validacion.
- El analisis de ajuste se basa en el gap entre recall de entrenamiento y validacion en CV para detectar senales de sobreajuste/subajuste.

In [27]:
# resumen compacto de resultados de tuning (salida corta para lectura rapida)
print('=' * 100)
print('RESUMEN COMPACTO | MEJORES PARAMETROS')
print('=' * 100)
print(best_params_df[['Escenario', 'Modelo', 'Best Recall CV (maligna)', 'Std Recall CV']].to_string(index=False))

print('\n' + '=' * 100)
print('RESUMEN COMPACTO | DELTAS OPTIMIZADO - BASE')
print('=' * 100)
print(df_delta.to_string(index=False))

print('\n' + '=' * 100)
print('RESUMEN COMPACTO | DIAGNOSTICO DE AJUSTE')
print('=' * 100)
print(df_fit_diag.to_string(index=False))

RESUMEN COMPACTO | MEJORES PARAMETROS
              Escenario              Modelo  Best Recall CV (maligna)  Std Recall CV
A - Todas las variables Regresion Logistica                    0.9789         0.0421
A - Todas las variables                 KNN                    0.9632         0.0488
    B - Top 3 variables Regresion Logistica                    0.9471         0.0333
    B - Top 3 variables                 KNN                    0.9578         0.0268

RESUMEN COMPACTO | DELTAS OPTIMIZADO - BASE
              Escenario              Modelo  Delta_Recall_CV_objetivo  Delta_Recall_Test_maligna  Delta_Precision_Test_maligna  Delta_F1_Test_maligna  Delta_Accuracy_Test  Delta_AUC_Test  Delta_FN_Test
A - Todas las variables                 KNN                    0.0160                    -0.0213                       -0.0014                -0.0115              -0.0075         -0.0011              1
A - Todas las variables Regresion Logistica                    0.0263                   

## 10. Tuning de hiperparametros: SVM con kernel RBF

- Se conserva el mismo split 80/20 estratificado y el mismo `StratifiedKFold` de 5 folds para mantener comparabilidad con LR y KNN.
- En SVM-RBF se afinan `C`, `gamma` y `class_weight`, porque controlan la complejidad del margen, la flexibilidad no lineal y el costo relativo de errores en la clase maligna.
- La seleccion sigue priorizando `recall` de la clase maligna (`Class=4`) para minimizar falsos negativos.

In [28]:
param_grid_svm = {
    'model__C': [0.1, 0.3, 1, 3, 10, 30, 100],
    'model__gamma': ['scale', 0.01, 0.03, 0.1, 0.3, 1],
    'model__class_weight': [None, 'balanced'],
    'model__kernel': ['rbf'],
    'model__probability': [True]
}

base_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', random_state=RANDOM_STATE, probability=True))
])

print('=' * 90)
print('CONFIGURACION DE TUNING (SVM RBF)')
print('=' * 90)
print('Metrica principal de seleccion: recall_maligna (pos_label=4)')
print(f'CV: StratifiedKFold con {cv_strategy.get_n_splits()} folds')
print(f'Combinaciones SVM aproximadas: {len(param_grid_svm["model__C"]) * len(param_grid_svm["model__gamma"]) * len(param_grid_svm["model__class_weight"])}')
print('Hiperparametros explorados: C, gamma, class_weight')
print('=' * 90)

CONFIGURACION DE TUNING (SVM RBF)
Metrica principal de seleccion: recall_maligna (pos_label=4)
CV: StratifiedKFold con 5 folds
Combinaciones SVM aproximadas: 84
Hiperparametros explorados: C, gamma, class_weight


In [29]:
from sklearn.metrics import confusion_matrix

rows_svm_best = []
rows_svm_compare = []
rows_svm_fit = []
rows_svm_top = []
rows_svm_errors = []

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    print('\n' + '=' * 90)
    print(f'TUNING SVM EN ESCENARIO: {nombre_escenario}')
    print('=' * 90)

    grid_svm = GridSearchCV(
        estimator=base_svm,
        param_grid=param_grid_svm,
        scoring=scorer_recall_maligna,
        cv=cv_strategy,
        n_jobs=N_JOBS,
        refit=True,
        return_train_score=True
    )
    grid_svm.fit(X_tr_esc, y_train)
    tuning_store[(nombre_escenario, 'SVM')] = grid_svm

    best_idx = grid_svm.best_index_
    best_mean = grid_svm.cv_results_['mean_test_score'][best_idx]
    best_std = grid_svm.cv_results_['std_test_score'][best_idx]
    best_train = grid_svm.cv_results_['mean_train_score'][best_idx]
    best_gap = best_train - best_mean

    base_cv_obj = cross_validate(
        modelos_base['SVM (RBF)'],
        X_tr_esc,
        y_train,
        cv=cv_strategy,
        scoring=scorer_recall_maligna,
        n_jobs=N_JOBS,
        return_train_score=True
    )
    base_cv_recall = base_cv_obj['test_score'].mean()
    base_train_recall = base_cv_obj['train_score'].mean()
    base_gap = base_train_recall - base_cv_recall

    base_model_fit = modelos_base['SVM (RBF)'].fit(X_tr_esc, y_train)
    tuned_model = grid_svm.best_estimator_

    base_test = evaluar_en_test(base_model_fit, X_te_esc, y_test)
    tuned_test = evaluar_en_test(tuned_model, X_te_esc, y_test)

    y_pred_base = base_model_fit.predict(X_te_esc)
    y_pred_tuned = tuned_model.predict(X_te_esc)
    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_test, y_pred_base, labels=LABELS_CLASE).ravel()
    tn_o, fp_o, fn_o, tp_o = confusion_matrix(y_test, y_pred_tuned, labels=LABELS_CLASE).ravel()

    rows_svm_best.append({
        'Escenario': nombre_escenario,
        'Modelo': 'SVM',
        'Best Recall CV (maligna)': round(best_mean, 4),
        'Std Recall CV': round(best_std, 4),
        'Best Params': str(grid_svm.best_params_)
    })

    rows_svm_compare.append({
        'Escenario': nombre_escenario,
        'Tipo': 'Base',
        'Recall_CV_maligna': round(base_cv_recall, 4),
        'Recall_Test_maligna': round(base_test['recall_maligna_test'], 4),
        'Precision_Test_maligna': round(base_test['precision_maligna_test'], 4),
        'F1_Test_maligna': round(base_test['f1_maligna_test'], 4),
        'Accuracy_Test': round(base_test['accuracy_test'], 4),
        'AUC_Test': round(base_test['auc_test'], 4),
        'FN_Test': base_test['fn_test']
    })
    rows_svm_compare.append({
        'Escenario': nombre_escenario,
        'Tipo': 'Optimizado',
        'Recall_CV_maligna': round(best_mean, 4),
        'Recall_Test_maligna': round(tuned_test['recall_maligna_test'], 4),
        'Precision_Test_maligna': round(tuned_test['precision_maligna_test'], 4),
        'F1_Test_maligna': round(tuned_test['f1_maligna_test'], 4),
        'Accuracy_Test': round(tuned_test['accuracy_test'], 4),
        'AUC_Test': round(tuned_test['auc_test'], 4),
        'FN_Test': tuned_test['fn_test']
    })

    rows_svm_fit.append({
        'Escenario': nombre_escenario,
        'Tipo': 'Base',
        'Recall_train_CV': round(base_train_recall, 4),
        'Recall_valid_CV': round(base_cv_recall, 4),
        'Gap_train_valid': round(base_gap, 4),
        'Diagnostico_ajuste': 'posible sobreajuste' if base_gap > 0.05 else ('posible subajuste' if base_cv_recall < 0.85 else 'estable')
    })
    rows_svm_fit.append({
        'Escenario': nombre_escenario,
        'Tipo': 'Optimizado',
        'Recall_train_CV': round(best_train, 4),
        'Recall_valid_CV': round(best_mean, 4),
        'Gap_train_valid': round(best_gap, 4),
        'Diagnostico_ajuste': 'posible sobreajuste' if best_gap > 0.05 else ('posible subajuste' if best_mean < 0.85 else 'estable')
    })

    rows_svm_errors.extend([
        {'Escenario': nombre_escenario, 'Tipo': 'Base', 'TN': int(tn_b), 'FP': int(fp_b), 'FN': int(fn_b), 'TP': int(tp_b)},
        {'Escenario': nombre_escenario, 'Tipo': 'Optimizado', 'TN': int(tn_o), 'FP': int(fp_o), 'FN': int(fn_o), 'TP': int(tp_o)}
    ])

    top_grid = pd.DataFrame(grid_svm.cv_results_)[[
        'param_model__C',
        'param_model__gamma',
        'param_model__class_weight',
        'mean_test_score',
        'std_test_score',
        'mean_train_score',
        'rank_test_score'
    ]].copy()
    top_grid = top_grid.sort_values(['mean_test_score', 'std_test_score', 'mean_train_score'], ascending=[False, True, True]).head(5)
    top_grid.insert(0, 'Escenario', nombre_escenario)
    top_grid.columns = ['Escenario', 'C', 'Gamma', 'Class_weight', 'Recall_valid_CV', 'Std_Recall_CV', 'Recall_train_CV', 'Rank']
    rows_svm_top.append(top_grid)

    print(f"Best Recall CV={best_mean:.4f} +/- {best_std:.4f}")
    print(f"Best Params   ={grid_svm.best_params_}")

svm_best_params_df = pd.DataFrame(rows_svm_best)
df_svm_compare = pd.DataFrame(rows_svm_compare)
df_svm_fit = pd.DataFrame(rows_svm_fit)
df_svm_errors = pd.DataFrame(rows_svm_errors)
df_svm_top = pd.concat(rows_svm_top, ignore_index=True)

print('\n' + '=' * 120)
print('TABLA: MEJORES HIPERPARAMETROS (SVM)')
print('=' * 120)
print(svm_best_params_df.to_string(index=False))

print('\n' + '=' * 120)
print('TABLA: SVM BASE VS OPTIMIZADO')
print('=' * 120)
print(df_svm_compare.to_string(index=False))

print('\n' + '=' * 120)
print('TABLA: EVIDENCIA DE AJUSTE EN SVM')
print('=' * 120)
print(df_svm_fit.to_string(index=False))

print('\n' + '=' * 120)
print('TABLA: ERRORES EN TEST (SVM)')
print('=' * 120)
print(df_svm_errors.to_string(index=False))

print('\n' + '=' * 120)
print('TABLA: TOP 5 CONFIGURACIONES SVM POR ESCENARIO')
print('=' * 120)
print(df_svm_top.to_string(index=False))


TUNING SVM EN ESCENARIO: A - Todas las variables
Best Recall CV=1.0000 +/- 0.0000
Best Params   ={'model__C': 0.1, 'model__class_weight': None, 'model__gamma': 1, 'model__kernel': 'rbf', 'model__probability': True}

TUNING SVM EN ESCENARIO: B - Top 3 variables
Best Recall CV=0.9735 +/- 0.0166
Best Params   ={'model__C': 3, 'model__class_weight': 'balanced', 'model__gamma': 'scale', 'model__kernel': 'rbf', 'model__probability': True}

TABLA: MEJORES HIPERPARAMETROS (SVM)
              Escenario Modelo  Best Recall CV (maligna)  Std Recall CV                                                                                                                     Best Params
A - Todas las variables    SVM                    1.0000         0.0000           {'model__C': 0.1, 'model__class_weight': None, 'model__gamma': 1, 'model__kernel': 'rbf', 'model__probability': True}
    B - Top 3 variables    SVM                    0.9735         0.0166 {'model__C': 3, 'model__class_weight': 'balanced', '

## 11. Evidencia de sobreajuste/subajuste y comparacion integrada

Para comparar LR, KNN y SVM bajo exactamente la misma metrica objetivo, se reconstruye una tabla integrada con `recall` de la clase maligna en CV, desempeno en test y gap `train-valid`. Asi se verifica si la mejora observada en validacion tambien se sostiene fuera de la muestra de entrenamiento.

In [30]:
modelos_integrados = {
    'Regresion Logistica': 'Regresion Logistica',
    'KNN': 'KNN (k=5)',
    'SVM': 'SVM (RBF)'
}

rows_best_all = []
rows_compare_all = []
rows_fit_diag_all = []

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    for modelo_tuning, modelo_base in modelos_integrados.items():
        base_cv_obj = cross_validate(
            modelos_base[modelo_base],
            X_tr_esc,
            y_train,
            cv=cv_strategy,
            scoring=scorer_recall_maligna,
            n_jobs=N_JOBS,
            return_train_score=True
        )
        base_cv_recall = base_cv_obj['test_score'].mean()
        base_train_recall = base_cv_obj['train_score'].mean()
        base_gap = base_train_recall - base_cv_recall
        base_diag = 'posible sobreajuste' if base_gap > 0.05 else ('posible subajuste' if base_cv_recall < 0.85 else 'estable')
        base_test = evaluar_en_test(modelos_base[modelo_base].fit(X_tr_esc, y_train), X_te_esc, y_test)

        tuned_grid = tuning_store[(nombre_escenario, modelo_tuning)]
        tuned_idx = tuned_grid.best_index_
        tuned_cv_recall = tuned_grid.cv_results_['mean_test_score'][tuned_idx]
        tuned_std = tuned_grid.cv_results_['std_test_score'][tuned_idx]
        tuned_train_recall = tuned_grid.cv_results_['mean_train_score'][tuned_idx]
        tuned_gap = tuned_train_recall - tuned_cv_recall
        tuned_diag = 'posible sobreajuste' if tuned_gap > 0.05 else ('posible subajuste' if tuned_cv_recall < 0.85 else 'estable')
        tuned_test = evaluar_en_test(tuned_grid.best_estimator_, X_te_esc, y_test)
        sostiene_en_test = 'si' if tuned_test['recall_maligna_test'] >= tuned_cv_recall - 0.03 else 'no'

        rows_best_all.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_tuning,
            'Best Recall CV (maligna)': round(tuned_cv_recall, 4),
            'Std Recall CV': round(tuned_std, 4),
            'Best Params': str(tuned_grid.best_params_)
        })

        rows_compare_all.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_tuning,
            'Tipo': 'Base',
            'Recall_CV_maligna': round(base_cv_recall, 4),
            'Recall_Test_maligna': round(base_test['recall_maligna_test'], 4),
            'Precision_Test_maligna': round(base_test['precision_maligna_test'], 4),
            'F1_Test_maligna': round(base_test['f1_maligna_test'], 4),
            'Accuracy_Test': round(base_test['accuracy_test'], 4),
            'AUC_Test': round(base_test['auc_test'], 4),
            'FN_Test': base_test['fn_test']
        })
        rows_compare_all.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_tuning,
            'Tipo': 'Optimizado',
            'Recall_CV_maligna': round(tuned_cv_recall, 4),
            'Recall_Test_maligna': round(tuned_test['recall_maligna_test'], 4),
            'Precision_Test_maligna': round(tuned_test['precision_maligna_test'], 4),
            'F1_Test_maligna': round(tuned_test['f1_maligna_test'], 4),
            'Accuracy_Test': round(tuned_test['accuracy_test'], 4),
            'AUC_Test': round(tuned_test['auc_test'], 4),
            'FN_Test': tuned_test['fn_test']
        })

        rows_fit_diag_all.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_tuning,
            'Tipo': 'Base',
            'Recall_train_CV': round(base_train_recall, 4),
            'Recall_valid_CV': round(base_cv_recall, 4),
            'Gap_train_valid': round(base_gap, 4),
            'Diagnostico_ajuste': base_diag,
            'Sostiene_en_test': 'referencia'
        })
        rows_fit_diag_all.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_tuning,
            'Tipo': 'Optimizado',
            'Recall_train_CV': round(tuned_train_recall, 4),
            'Recall_valid_CV': round(tuned_cv_recall, 4),
            'Gap_train_valid': round(tuned_gap, 4),
            'Diagnostico_ajuste': tuned_diag,
            'Sostiene_en_test': sostiene_en_test
        })

best_params_df_all = pd.DataFrame(rows_best_all)
df_compare_all = pd.DataFrame(rows_compare_all)
df_fit_diag_all = pd.DataFrame(rows_fit_diag_all)

rows_delta_all = []
for (escenario, modelo), grp in df_compare_all.groupby(['Escenario', 'Modelo']):
    base_row = grp[grp['Tipo'] == 'Base'].iloc[0]
    opt_row = grp[grp['Tipo'] == 'Optimizado'].iloc[0]
    rows_delta_all.append({
        'Escenario': escenario,
        'Modelo': modelo,
        'Delta_Recall_CV_maligna': round(opt_row['Recall_CV_maligna'] - base_row['Recall_CV_maligna'], 4),
        'Delta_Recall_Test_maligna': round(opt_row['Recall_Test_maligna'] - base_row['Recall_Test_maligna'], 4),
        'Delta_Precision_Test_maligna': round(opt_row['Precision_Test_maligna'] - base_row['Precision_Test_maligna'], 4),
        'Delta_F1_Test_maligna': round(opt_row['F1_Test_maligna'] - base_row['F1_Test_maligna'], 4),
        'Delta_Accuracy_Test': round(opt_row['Accuracy_Test'] - base_row['Accuracy_Test'], 4),
        'Delta_AUC_Test': round(opt_row['AUC_Test'] - base_row['AUC_Test'], 4),
        'Delta_FN_Test': int(opt_row['FN_Test'] - base_row['FN_Test'])
    })
df_delta_all = pd.DataFrame(rows_delta_all)

df_optimized_only = df_compare_all[df_compare_all['Tipo'] == 'Optimizado'].copy()
df_optimized_only['Sostiene_en_test'] = np.where(
    df_optimized_only['Recall_Test_maligna'] >= df_optimized_only['Recall_CV_maligna'] - 0.03,
    'si',
    'no'
)
df_optimized_only = df_optimized_only.sort_values(['Escenario', 'Recall_Test_maligna', 'FN_Test', 'AUC_Test'], ascending=[True, False, True, False])

print('=' * 140)
print('TABLA INTEGRADA: MEJORES HIPERPARAMETROS (LR + KNN + SVM)')
print('=' * 140)
print(best_params_df_all.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA INTEGRADA: COMPARACION BASE VS OPTIMIZADO (METRICA OBJETIVO = RECALL MALIGNA)')
print('=' * 140)
print(df_compare_all.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA INTEGRADA: DELTAS (OPTIMIZADO - BASE)')
print('=' * 140)
print(df_delta_all.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA INTEGRADA: DIAGNOSTICO DE AJUSTE')
print('=' * 140)
print(df_fit_diag_all.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA INTEGRADA: CANDIDATOS OPTIMIZADOS ORDENADOS POR RECALL EN TEST')
print('=' * 140)
print(df_optimized_only.to_string(index=False))

TABLA INTEGRADA: MEJORES HIPERPARAMETROS (LR + KNN + SVM)
              Escenario              Modelo  Best Recall CV (maligna)  Std Recall CV                                                                                                                         Best Params
A - Todas las variables Regresion Logistica                    0.9789         0.0421 {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 2000, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
A - Todas las variables                 KNN                    0.9632         0.0488                                                {'model__metric': 'euclidean', 'model__n_neighbors': 9, 'model__weights': 'uniform'}
A - Todas las variables                 SVM                    1.0000         0.0000               {'model__C': 0.1, 'model__class_weight': None, 'model__gamma': 1, 'model__kernel': 'rbf', 'model__probability': True}
    B - Top 3 variables Regresion Logistica                    0.9471     

## 12. Comparacion final, seleccion del mejor modelo y texto base para documento (Semana 3)

### Criterios de seleccion aplicados
1. Maximizar `Recall_Test_maligna` (objetivo clinico principal).
2. Minimizar `FN_Test` para reducir falsos negativos.
3. Verificar estabilidad (`Sostiene_en_test = si`).
4. Usar precision, F1 y AUC como trade-offs secundarios.

### Resultado de comparacion final (modelos optimizados)
Se uso la tabla integrada `df_optimized_only` (exportada como `19_optimized_only.csv`) para comparar LR, KNN y SVM en ambos escenarios bajo exactamente el mismo protocolo de validacion.

Hallazgos clave:
- `SVM optimizado - Escenario A` logra `Recall_Test_maligna = 1.0000` y `FN_Test = 0`.
- `SVM optimizado - Escenario B` queda como segundo mejor por sensibilidad (`Recall_Test_maligna = 0.9787`, `FN_Test = 1`).
- `LR optimizada - Escenario A` es alternativa competitiva por balance (`Recall_Test_maligna = 0.9574`, `AUC_Test = 0.9944`), pero no supera a SVM-A en falsos negativos.
- `KNN` no muestra mejora consistente tras tuning en los dos escenarios.

### Modelo final seleccionado
**Seleccion final: SVM optimizado con kernel RBF en Escenario A (todas las variables).**

Justificacion tecnica:
- Es el unico candidato con cero falsos negativos en test (`FN_Test = 0`).
- Mantiene sensibilidad maxima para la clase maligna (`Recall_Test_maligna = 1.0000`).
- La mejora en validacion se sostiene en test (`Sostiene_en_test = si`).
- El costo asociado es una reduccion de precision/AUC frente a LR, trade-off aceptable dada la prioridad clinica de no omitir casos malignos.

### Discusion tecnica de bias/variance
- No se observan senales fuertes de sobreajuste en los modelos optimizados: los gaps `train-valid` son pequenos.
- En SVM-A optimizado, el gap es 0.0000, consistente con comportamiento estable en validacion.
- El patron dominante no es sobreajuste clasico, sino un cambio de umbral implicito: mayor sensibilidad con mas falsos positivos relativos, lo cual desplaza el balance sesgo-varianza hacia menor sesgo para la clase positiva.

### Nota de consistencia notebook-documento
La seleccion final y su justificacion deben quedar identicas en el documento de Semana 3, usando como fuente las tablas exportadas `16_compare_all.csv`, `18_fit_diag_all.csv` y `19_optimized_only.csv`.

## 13. Exportacion de tablas a CSV

Las tablas principales del notebook se exportan a una carpeta de CSVs para uso en el documento, comparaciones externas o respaldo de resultados. Se usa `utf-8-sig` para facilitar apertura en Excel.

In [31]:
candidatos_export = [
    Path('data/resultados_semana3_csv'),
    Path('../data/resultados_semana3_csv')
]
export_dir = None
for ruta in candidatos_export:
    if ruta.parent.exists():
        export_dir = ruta
        break
if export_dir is None:
    export_dir = candidatos_export[0]
export_dir.mkdir(parents=True, exist_ok=True)

tablas_export = {
    '01_tabla_distribucion_clases.csv': tabla_dist,
    '02_cv_results_global.csv': df_cv_results,
    '03_cv_results_escenario_A.csv': df_cv_results[df_cv_results['Escenario'] == 'A - Todas las variables'].reset_index(drop=True),
    '04_cv_results_escenario_B.csv': df_cv_results[df_cv_results['Escenario'] == 'B - Top 3 variables'].reset_index(drop=True),
    '05_cv_vs_test.csv': df_cv_vs_test,
    '06_best_params_lr_knn.csv': best_params_df,
    '07_compare_lr_knn.csv': df_compare,
    '08_delta_lr_knn.csv': df_delta,
    '09_fit_diag_lr_knn.csv': df_fit_diag,
    '10_best_params_svm.csv': svm_best_params_df,
    '11_compare_svm.csv': df_svm_compare,
    '12_fit_svm.csv': df_svm_fit,
    '13_errors_svm.csv': df_svm_errors,
    '14_top_svm.csv': df_svm_top,
    '15_best_params_all.csv': best_params_df_all,
    '16_compare_all.csv': df_compare_all,
    '17_delta_all.csv': df_delta_all,
    '18_fit_diag_all.csv': df_fit_diag_all,
    '19_optimized_only.csv': df_optimized_only
}

manifest_rows = []
for nombre_archivo, tabla in tablas_export.items():
    destino = export_dir / nombre_archivo
    tabla.to_csv(destino, index=False, encoding='utf-8-sig')
    manifest_rows.append({
        'archivo': nombre_archivo,
        'filas': int(tabla.shape[0]),
        'columnas': int(tabla.shape[1]),
        'ruta': str(destino)
    })

df_manifest = pd.DataFrame(manifest_rows)
df_manifest.to_csv(export_dir / '00_manifest.csv', index=False, encoding='utf-8-sig')

print('=' * 120)
print('EXPORTACION DE TABLAS COMPLETADA')
print('=' * 120)
print(f'Carpeta destino: {export_dir.resolve()}')
print(df_manifest.to_string(index=False))

EXPORTACION DE TABLAS COMPLETADA
Carpeta destino: C:\Users\rjbar\OneDrive\Documents\GitHub\PRY2-MD\data\resultados_semana3_csv
                         archivo  filas  columnas                                                            ruta
01_tabla_distribucion_clases.csv      2         3 ..\data\resultados_semana3_csv\01_tabla_distribucion_clases.csv
        02_cv_results_global.csv      6         7         ..\data\resultados_semana3_csv\02_cv_results_global.csv
   03_cv_results_escenario_A.csv      3         7    ..\data\resultados_semana3_csv\03_cv_results_escenario_A.csv
   04_cv_results_escenario_B.csv      3         7    ..\data\resultados_semana3_csv\04_cv_results_escenario_B.csv
               05_cv_vs_test.csv      6        12                ..\data\resultados_semana3_csv\05_cv_vs_test.csv
       06_best_params_lr_knn.csv      4         5        ..\data\resultados_semana3_csv\06_best_params_lr_knn.csv
           07_compare_lr_knn.csv      8        10            ..\data\result